# QYYTMB Processing

- Whole genome sequence data; Illumina and Nanopore.
- Shipped and submitted to Plasmidsaurus 02/06/2026.
- Data available on: 02/23/2026.
- Started procecessing on: 02/25/2026.

## Description of sequencing samples
3 isolates each of strain stocks ANLstock.ACN3667 and ANLstock.ACN3749

## Steps
1. Download, data organization, file renaming
2. Creating seqsamples and cross-checking in LIMS.
3. QA/QC
4. Breseq Pipeline and Breseq analysis
   1. Reference: ACN3500; subsampled to 300x
   2. Breseq summary (with copy number calculation over ver cassette) and breseq comparison for all samples.

## 0. Set-up

In [ ]:
## Make sure running in aisynbio_env

In [1]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

## 1. Download, file organization, renaming
- Create seqorder name and makedir into reception folder (/synbio/ai_synbio_data/experimental_data/downloads - should this be temporary?). Also make homedir to copy and view analysis reports.
- Download data into folder.
- Spot-check if duplicate read ID problem is fixed.
- Create new seqorder folder in experimental_data/sequencing_data/ and respective libraries.
- Copy all nanopore fastqs into long lib, renaming in the process.
- Copy all illumina fastqs into short lib, renaming in the process.
Creating sample seqs and cross-checking with LIMS


In [4]:
# Create seqorder name, make reception folder and seqorder analysis folder in nspahr homedir

from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample
import os

item_code = 'QYYTMB'
seqorder_name = create_seqorder_name(item_code)

reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
os.makedirs(reception_dir)
print(reception_dir)

home_dir = '/storage/nspahr/lib_analysis/' + seqorder_name
os.makedirs(home_dir, exist_ok=True) 

/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-02-13_QYYTMB


In [5]:
# Download data into folder

from aisynbiopipeline.workflows.plasmidsaurus import get_access_token, download_results, get_credentials

CLIENT_ID = get_credentials("PLASMIDSAURUS_CLIENT_ID")
CLIENT_SECRET = get_credentials("PLASMIDSAURUS_CLIENT_SECRET")
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)
download_results(item_code, access_token, reception_dir)

ITEM QYYTMB
{'code': 'QYYTMB',
 'done_date': '2026-02-13T21:19:24.146081+00:00',
 'gross': 1080.0,
 'order_name': '',
 'product_name': 'hybrid_extraction',
 'quantity': 6,
 'status': 'complete'}



DOWNLOADING RESULTS FOR QYYTMB 


File downloaded successfully: /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-02-13_QYYTMB/QYYTMB_results.zip
Unzipping /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-02-13_QYYTMB/QYYTMB_results.zip to /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-02-13_QYYTMB/QYYTMB_results
DOWNLOADING READS FOR QYYTMB 


File downloaded successfully: /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-02-13_QYYTMB/QYYTMB_reads.zip
Unzipping /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-02-13_QYYTMB/QYYTMB_reads.zip to /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-02-13_QYYTMB/QYYTMB_reads


In [7]:
!ls /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-02-13_QYYTMB/QYYTMB_reads

QYYTMB_1_ANLstock.ACN3749.colony1_illumina_R1.fastq.gz
QYYTMB_1_ANLstock.ACN3749.colony1_illumina_R2.fastq.gz
QYYTMB_1_ANLstock.ACN3749.colony1_nanopore.fastq.gz
QYYTMB_2_ANLstock.ACN3749.colony2_illumina_R1.fastq.gz
QYYTMB_2_ANLstock.ACN3749.colony2_illumina_R2.fastq.gz
QYYTMB_2_ANLstock.ACN3749.colony2_nanopore.fastq.gz
QYYTMB_3_ANLstock.ACN3749.colony3_illumina_R1.fastq.gz
QYYTMB_3_ANLstock.ACN3749.colony3_illumina_R2.fastq.gz
QYYTMB_3_ANLstock.ACN3749.colony3_nanopore.fastq.gz
QYYTMB_4_ANLstock.ACN3667.colony1_illumina_R1.fastq.gz
QYYTMB_4_ANLstock.ACN3667.colony1_illumina_R2.fastq.gz
QYYTMB_4_ANLstock.ACN3667.colony1_nanopore.fastq.gz
QYYTMB_5_ANLstock.ACN3667.colony2_illumina_R1.fastq.gz
QYYTMB_5_ANLstock.ACN3667.colony2_illumina_R2.fastq.gz
QYYTMB_5_ANLstock.ACN3667.colony2_nanopore.fastq.gz
QYYTMB_6_ANLstock.ACN3667.colony3_illumina_R1.fastq.gz
QYYTMB_6_ANLstock.ACN3667.colony3_illumina_R2.fastq.gz
QYYTMB_6_ANLstock.ACN3667.colony3_nanopore.fastq.gz


In [ ]:
### TODO: How to ensure that archive was successfully unzipped?? Pipeline log file into home_dir?

In [9]:
# Spot-check if duplicate read ID problem is fixed

plasmidsaurus_read_folder_name = item_code + '_reads'
os.listdir(os.path.join(reception_dir, plasmidsaurus_read_folder_name))

['QYYTMB_4_ANLstock.ACN3667.colony1_nanopore.fastq.gz',
 'QYYTMB_4_ANLstock.ACN3667.colony1_illumina_R1.fastq.gz',
 'QYYTMB_4_ANLstock.ACN3667.colony1_illumina_R2.fastq.gz',
 'QYYTMB_3_ANLstock.ACN3749.colony3_nanopore.fastq.gz',
 'QYYTMB_3_ANLstock.ACN3749.colony3_illumina_R1.fastq.gz',
 'QYYTMB_3_ANLstock.ACN3749.colony3_illumina_R2.fastq.gz',
 'QYYTMB_6_ANLstock.ACN3667.colony3_nanopore.fastq.gz',
 'QYYTMB_6_ANLstock.ACN3667.colony3_illumina_R1.fastq.gz',
 'QYYTMB_6_ANLstock.ACN3667.colony3_illumina_R2.fastq.gz',
 'QYYTMB_1_ANLstock.ACN3749.colony1_nanopore.fastq.gz',
 'QYYTMB_1_ANLstock.ACN3749.colony1_illumina_R1.fastq.gz',
 'QYYTMB_1_ANLstock.ACN3749.colony1_illumina_R2.fastq.gz',
 'QYYTMB_5_ANLstock.ACN3667.colony2_nanopore.fastq.gz',
 'QYYTMB_5_ANLstock.ACN3667.colony2_illumina_R1.fastq.gz',
 'QYYTMB_5_ANLstock.ACN3667.colony2_illumina_R2.fastq.gz',
 'QYYTMB_2_ANLstock.ACN3749.colony2_nanopore.fastq.gz',
 'QYYTMB_2_ANLstock.ACN3749.colony2_illumina_R1.fastq.gz',
 'QYYTMB_2_ANLs

In [12]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, 'QYYTMB_6_ANLstock.ACN3667.colony3_illumina_R2.fastq.gz')} | head

@LH01080:8:23KYCNLT4:1:1101:2869:1042 2:N:0:TTAACCTTCG+TGTCTGGCCT
GTATTAATCCATTTCAATAATTTGCTGGCCAGTAATTCTTGCTGCTGTGCAACTTTAATTTCAGCAAGAGGTGCTTGCTGGCTCCGCATACGCTCAATCAATTGAGCATAATTTGCCTGAATTTCTGGCATGGAAAGCTGCATTGCCAGAT
+
II9IIII9IIIIIIIIIIII9IIIII9III9IIIIIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII*IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIII9III
@LH01080:8:23KYCNLT4:1:1101:7982:1042 2:N:0:TTAACCTTCG+TGTCTGGCCT
ACTGCGTTCCACAATACCTGCGATATCTTTGGGTACGTTGACTGCACCACCAGAATTAATATCGTTCGTCACGGCCTGTAAACCGCTGATCAAACCTAACATATAGATGGTCTGGTCCAGATCATTACGCATGGTTGGACAAGAATCCCCC
+
IIIIIIIIIIIIIIIIIIII*IIIIIIIIIIIIIIIIIIIIIIIIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII9IIIIIII9IIIIIIIIIIIIIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@LH01080:8:23KYCNLT4:1:1101:9859:1042 2:N:0:TTAACCTTCG+TNTCTGGCCT
CTATTTGGGTTAATAGTATTGAAAGCATTGTCATCCTTATCAAATTTAGGATTAGGTTCCACACCATATCTATAAATTGTCTGTCTCTTATACACATCTGACGCTGCCGACGATGTCTGGCCTGTGTAGATCTCGGTGGTCGCCGTATCAT

gzip: stdout: Broken pipe


In [13]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, 'QYYTMB_6_ANLstock.ACN3667.colony3_illumina_R2.fastq.gz')} | grep "@LH01080:8:23KYCNLT4:1:1101:9859:1042 2:N:0:TTAACCTTCG+TNTCTGGCCT"


@LH01080:8:23KYCNLT4:1:1101:9859:1042 2:N:0:TTAACCTTCG+TNTCTGGCCT


**Comment:**

- In this spot check, only found the read ID once in this file. I assume this means that we are not dealing with the read duplication problem seen initially in order P4CYGL.

In [14]:
from aisynbiopipeline.workflows.fastq_utils import create_manifest, parse_illumina_fastq_filename

folder = os.path.join(reception_dir, plasmidsaurus_read_folder_name)

manifest = create_manifest(folder, platform='plasmidsaurus_hybrid')
manifest

,sample_name,nanopore_fastq,fwd_fastq,rvs_fastq
3,QYYTMB_1_ANLstock.ACN3749.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
5,QYYTMB_2_ANLstock.ACN3749.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,QYYTMB_3_ANLstock.ACN3749.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
0,QYYTMB_4_ANLstock.ACN3667.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,QYYTMB_5_ANLstock.ACN3667.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,QYYTMB_6_ANLstock.ACN3667.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [15]:
# Plasmidsaurus provides a sort of sample manifest for download from the seqorder page. (not available with read download through API).
# Downloaded to my laptop, uploaded to home_dir, now copying to reception dir.

import shutil

shutil.copy2(os.path.join(home_dir, f'{item_code}-summary-report.csv'), reception_dir)

'/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-02-13_QYYTMB/QYYTMB-summary-report.csv'

In [16]:
os.listdir(reception_dir)

['QYYTMB_results.zip',
 'QYYTMB_results',
 'QYYTMB_reads.zip',
 'QYYTMB_reads',
 'QYYTMB-summary-report.csv']

In [17]:
# Create new seqorder folder in experimental_data/sequencing_data/ and libraries

seqorder = SeqOrder(seqorder_name, create=True)
short = Library(seqorder, 'Illumina', create=True)
long = Library(seqorder, 'Nanopore', create=True)

In [18]:
# Identify Illumina/ Nanopore reads and copy into respective library folder

from pathlib import Path
import shutil

reads_path = Path(os.path.join(reception_dir, f'{item_code}_reads'))
i_pattern = "*illumina*.fastq.gz"
n_pattern = "*nanopore*.fastq.gz"
i_files = list(reads_path.glob(i_pattern))
n_files = list(reads_path.glob(n_pattern))

def rename_plasmidsaurus_read_file(file_name):
    aisynbio_filename = ('_').join(file_name.split('_')[2:])
    return aisynbio_filename

for file in i_files:
    plasmidsaurus_basename = os.path.basename(file)
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

for file in n_files:
    plasmidsaurus_basename = os.path.basename(file)
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, long.path/'received'/aisynbio_basename)

## 2. Cross-checking this Plasmidsaurus order seqsamples in LIMS and creating SeqSamples.

In [19]:
short_manifest = short.create_manifest('received')
short_manifest

,sample_name,R1,R2
0,ANLstock.ACN3667.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,ANLstock.ACN3667.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,ANLstock.ACN3667.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,ANLstock.ACN3749.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,ANLstock.ACN3749.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
5,ANLstock.ACN3749.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [20]:
# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)

%run util_simple.py

✓ LIMS API loaded successfully


In [ ]:
# Run a manual LIMS mirror db sync

from aisynbiopipeline.limsapi.sync import sync_all_sheets

sync_all_sheets()

In [21]:
# Cross-checking seq sample measurement names

thisExpLIMSseqsamples_short = query_lims(
    'Measurements',
    filters={'Experiment': 'strain_stocks', 'Type': 'Short_DNA_reads'}
)['Name'].to_list()

thisExpLIMSseqsamples_long = query_lims(
    'Measurements',
    filters={'Experiment': 'strain_stocks', 'Type': 'Long_DNA_reads'}
)['Name'].to_list()

print(f"Are all short Plasmidsaurus seqsamples from order {item_code} in the LIMS Measurements table?")
print(all([x in thisExpLIMSseqsamples_short for x in short_manifest['sample_name']]))

long_manifest = long.create_manifest('received')
print(f"Are all Plasmidsaurus long seqsamples from order {item_code} in the LIMS Measurements table?")
print(all([x in thisExpLIMSseqsamples_long for x in long_manifest['sample_name']]))

Are all short Plasmidsaurus seqsamples from order QYYTMB in the LIMS Measurements table?
False
Are all Plasmidsaurus long seqsamples from order QYYTMB in the LIMS Measurements table?
False


In [22]:
# Creating batch (list) of short seqsamples for this seqorder

seqsamples = [SeqSample(short, row['sample_name']) for _, row in short_manifest.iterrows()]

## 3. Short reads: QA/QC

- fastp (Celery): Must start running workers first
- MultiQC

In [23]:
# Create the trimmed subfolder
short.create_subfolder('trimmed')

In [ ]:
## To run fastp celery worksers, activate micromamba, then call worker script:
"""
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.fastp_task 1
"""

In [ ]:
## fastp workers running:

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 1

In [24]:
# Where should this code go?

from pathlib import Path

def get_fastp_params(library, seqsample):

    fwd_in_path = seqsample.received[0]
    fwd_in_file = os.path.basename(fwd_in_path)
    fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    fwd_out_path = os.path.join(library.path, 'trimmed', fwd_out_file)
    rvs_in_path = seqsample.received[1]
    rvs_in_file = os.path.basename(rvs_in_path)
    rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    rvs_out_path = os.path.join(library.path, 'trimmed', rvs_out_file)
    
    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    fastp_params = {
        'path_to_fwd': norm(fwd_in_path),
        'path_to_rev': norm(rvs_in_path),
        'path_to_fwd_out': norm(fwd_out_path),
        'path_to_rev_out': norm(rvs_out_path),
        'threads': 16,
        'polyG':5
    }

    return fastp_params

In [25]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks:

results = []

for sample in seqsamples:
    result = client.send_task(
        'fastp.run',
        kwargs=get_fastp_params(short, sample),
        queue='fastp'
    )
    results.append(result)

In [27]:
for i in results:
    print(i.status)

SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS


In [ ]:
all([(r.status=='SUCCESS') for r in results])

In [28]:
from aisynbiopipeline.workflows.read_qc import run_multiqc
import shutil

multiqc_report = run_multiqc(short.path / 'trimmed')
multiqc_report_dir = os.path.dirname(multiqc_report)
multiqc_report_file = os.path.basename(multiqc_report)
dst_multiqc_report_file = os.path.join(home_dir, 'trimmed_' + multiqc_report_file)
shutil.copy(multiqc_report, dst_multiqc_report_file)


/// ]8;id=109185;https://multiqc.info\MultiQC]8;;\ v1.32

     version_check | MultiQC Version v1.33 now available!
       file_search | Search path: /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-02-13_QYYTMB/Plasmidsaurus_2026-02-13_QYYTMB_Illumina/trimmed


        searching | ████████████████████████████████████████ 100% 24/24                                                   html

             fastp | Found 6 reports
     write_results | Data        : multiqc_data
     write_results | Report      : multiqc_report.html
           multiqc | MultiQC complete


'/storage/nspahr/lib_analysis/Plasmidsaurus_2026-02-13_QYYTMB/trimmed_multiqc_report.html'

## 4. Short reads: Breseq

In [29]:
from aisynbiopipeline.workflows.reference_utils import list_reference_genomes

list_reference_genomes()

['ADP1_Neidle_CDM.gbk',
 'ACN2821_CDM.gbk',
 'ACN2586_NSS.gbk',
 'ACN3500_IRZ.gbk',
 'ACN3500_NSS.gbk']

In [ ]:
## To run breseq celery worksers, activate micromamba, then call worker script:
"""
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.breseq_task 1
"""

In [ ]:
# Two breseq workers are running:

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 1
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 2

In [30]:
# Create breseq dir

short.create_subfolder('breseq')

In [31]:
short_manifest = short.create_manifest('received')
short_manifest

,sample_name,R1,R2
0,ANLstock.ACN3667.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,ANLstock.ACN3667.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,ANLstock.ACN3667.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,ANLstock.ACN3749.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,ANLstock.ACN3749.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
5,ANLstock.ACN3749.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [32]:
# Where should this code go?

# Specifies and assigns breseq parameters

def define_breseq_params(seqsample, ref_filename, poly=True, fold_coverage=300, num_processors=4):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': ref_filename,
        'polymorphism_prediction': poly,
        'limit_fold_coverage': fold_coverage,
        'num_processors': num_processors
    }
    return breseq_params

In [33]:
# Submit tasks

results = []

for sample in seqsamples:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_NSS.gbk'),
        queue='breseq'
    )
    results.append(result)    

In [34]:
sum([(r.status=='SUCCESS') for r in results])

6

In [36]:
for i in results:
    print(i.status)
for i in results:
    print(i.result['output']) ## Check version_name for breseq run

SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-02-13_QYYTMB/Plasmidsaurus_2026-02-13_QYYTMB_Illumina/breseq/ANLstock.ACN3667.colony1/breseq_df1972644b
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-02-13_QYYTMB/Plasmidsaurus_2026-02-13_QYYTMB_Illumina/breseq/ANLstock.ACN3667.colony2/breseq_df1972644b
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-02-13_QYYTMB/Plasmidsaurus_2026-02-13_QYYTMB_Illumina/breseq/ANLstock.ACN3667.colony3/breseq_df1972644b
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-02-13_QYYTMB/Plasmidsaurus_2026-02-13_QYYTMB_Illumina/breseq/ANLstock.ACN3749.colony1/breseq_df1972644b
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-02-13_QYYTMB/Plasmidsaurus_2026-02-13_QYYTMB_Illumina/breseq/ANLstock.ACN3749.colony2/breseq_df1972644b
/s

## 5. Breseq analysis

- Create breseq objects for each seqsample and aggregate summary counts.
- Generate mutation table for all samples and write to csv for import into Google Sheets.

In [37]:
from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path, genomic_region_from_features

genome3500 = os.path.join(get_ref_genomes_path(), 'ACN3500_NSS.gbk')

def get_region_parameter(genbank_file, feature_first, feature_last):
    genome, start, stop = genomic_region_from_features(genbank_file, feature_first, feature_last)
    region = genome + ":" + str(start) + "-" + str(stop)
    return region


In [49]:
get_region_parameter(genome3500, 'omega KmR cassette', 'verR')

'ACN3500_NSS:941311-950626'

In [38]:
from aisynbiopipeline.workflows.breseq import Breseq

def create_breseq_summary(seqsample_batch, version_name, output_path, regions=None):
    
    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)
    
    rows = []
    
    for b in breseq_objects:
        
        row = {}
        
        try:
            b.count_reads()
            b.count_mutations()
            b.avg_coverage
            if regions:
                for key, value in regions.items():
                    b.get_region_average_coverage(value)
        except Exception as e:
            row.update({'seqsample': getattr(b, 'title', None)})
            # row.update(parse_seqsample_name(getattr(b, 'title', None)))
            row.update({'error': str(e),
                        'input_read_count': None,
                        'used_read_count': None,
                        'mapped_read_count': None,
                        'consensus_mutation_count': None,
                        'polymorphism_mutation_count': None,
                        'average_cov': None,}
                        )
            if regions:
                for key, value in regions.items():
                    row.update({key: None})
            rows.append(row)
            print(f"Error loading Breseq from {b.title}: {e}")
            continue
    
        row['seqsample'] = getattr(b, 'title', None)
        # row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row['error'] = None
        row['input_read_count'] = getattr(b, 'input_read_count', None)
        row['used_read_count'] = getattr(b, 'used_read_count', None)
        row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
        row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
        row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)
        row['average_cov'] = getattr(b, 'avg_coverage', None)
        if regions:
            for key, value in regions.items():
                row[key] = getattr(b, 'get_region_average_coverage', None)(value)
        rows.append(row)
    
    breseq_summary = pd.DataFrame(rows)
    
    if regions:
        for key, value in regions.items():
            breseq_summary[key+'_CN'] = breseq_summary[key]/breseq_summary['average_cov']

    # Write breseq run summary to csv
    breseq_summary.to_csv(output_path)
    
    return breseq_summary

In [39]:
def create_html_comparison(seqsample_batch, version_name, output_path):

    from aisynbiopipeline.workflows.breseq import compare_gdiff

    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)

    reference = b.params.reference
    gdiffs = [b.gd_file for b in breseq_objects]

    table_format = 'html'
    html = compare_gdiff(reference, output_path, gdiffs, format=table_format)

    return html

In [47]:
# These strains are supposed to have the 6007 bp promoter deletion, including verR.
# Therefore, defining 'ver cassette' as verB through omega KmR cassette. 

regions = {
    # 'dgoA-Star': get_region_parameter(genome2821, 'dgoA-optimized-ADP1', 'dgoA-optimized-ADP1'),
    'ver_cassette': get_region_parameter(genome3500, 'omega KmR cassette', 'verB')
}

regions

{'ver_cassette': 'ACN3500_NSS:941311-949904'}

In [48]:
regions_to_include = ['ver_cassette']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

breseq_version_name = 'breseq_df1972644b'

breseq_folder = home_dir + '/' + breseq_version_name
os.makedirs(breseq_folder, exist_ok=True)

create_breseq_summary(seqsamples, breseq_version_name, os.path.join(breseq_folder, 'mutation_summary.csv'), regions=regions_sub)
create_html_comparison(seqsamples, breseq_version_name, os.path.join(breseq_folder, 'mutation_comparison.html'))

breseq 0.39.0     http://barricklab.org/breseq

Active Developers: Barrick JE, Deatherage DE
Contact:           <jeffrey.e.barrick@gmail.com>

breseq is free software; you can redistribute it and/or modify it under the
terms the GNU General Public License as published by the Free Software 
Foundation; either version 2, or (at your option) any later version.

Copyright (c) 2008-2010 Michigan State University
Copyright (c) 2011-2022 The University of Texas at Austin

If you use breseq in your research, please cite:

  Deatherage, D.E., Barrick, J.E. (2014) Identification of mutations
  in laboratory-evolved microbes from next-generation sequencing
  data using breseq. Methods Mol. Biol. 1151: 165–188.

If you use structural variation (junction) predictions, please cite:

  Barrick, J.E., Colburn, G., Deatherage D.E., Traverse, C.C.,
  Strand, M.D., Borges, J.J., Knoester, D.B., Reba, A., Meyer, A.G. 
  (2014) Identifying structural variation in haploid microbial genomes 
  from short-rea

'/storage/nspahr/lib_analysis/Plasmidsaurus_2026-02-13_QYYTMB/breseq_df1972644b/mutation_comparison.html'

In [44]:
# Symlink to library breseq folder
output_symlinks_dir = os.path.join(breseq_folder, 'symlink_to_library_breseq_folder')
os.makedirs(output_symlinks_dir)

path_to_folder = short.path / 'breseq'
dst = os.path.join(output_symlinks_dir, 'breseq')
os.symlink(path_to_folder, dst)